- 简单对比：https://github.com/volcengine/verl/blob/main/recipe/retool/run_qwen2_7b_sft.sh
    - 单机四卡的配置
    - 几乎完全相同的曲线（losses）
- fsdp2
    - 使用 PyTorch 原生 (v2.4+) 的 fully_shard API。
    - 它将模型的 参数 (Parameters)、梯度 (Gradients) 和 优化器状态 (Optimizer States) 切分到所有可用的 GPU 上

### torchrun

- `master-addr`, `master-port`, `node-rank`
    - `master-addr`: 主节点（Rank 0 节点）的 IP 地址或主机名。所有参与训练的节点（机器）都需要连接到这个地址，以便进行梯度的同步和广播。它是分布式通信的协调中心。
    - `master-port`: 主节点上用于分布式通信的空闲端口号。用于建立节点间的通信连接。
    - `node-rank`: 当前机器（节点）在集群中的全局序号索引。
- 单机（多卡）：`--standalone` 时，不需要指定 `--master-addr`和`--master-port`
    - torchrun 会自己用 127.0.0.1 和一个空闲端口。(pytorch 默认 29500)
- 多机（多卡）：对于所有 node，master_addr 和 master_port 都填同一个；

```
# 多机（示例：2 台，每台 8 卡）
# 节点0：
torchrun --nnodes=2 --nproc_per_node=8 --node_rank=0 \
         --master_addr=node0_host --master_port=29500 train.py
# 节点1：
torchrun --nnodes=2 --nproc_per_node=8 --node_rank=1 \
         --master_addr=node0_host --master_port=29500 train.py
```

### fsdp vs. fsdp2

- references
    - https://huggingface.co/docs/accelerate/concept_guides/fsdp1_vs_fsdp2
    - https://zhuanlan.zhihu.com/p/1929971661786048218
    - https://zhuanlan.zhihu.com/p/1929115059113693341
- FSDP（俗称 FSDP1） vs FSDP2（fully_shard）
    - fsdp：进入模块时做整块权重 all-gather（FlatParameter），算完释放；反向阶段按模块 reduce-scatter。
    - FSDP2（fully_shard 新 API）：逐参数 DTensor + 算子级按需 JIT all-gather/快速 reshard，副本存留窗口更窄、可重叠机会更多。

### param_offload、optimizer_offload

```
actor.fsdp_config.param_offload=True
actor.fsdp_config.optimizer_offload=True
```

- Then some workers keep params on CPU while others already sharded to GPU → leads to asymmetric memory layout.